# VitaStress — Passi 1 e 2: tabella e classificatore binario
Parte 1: la tabella degli autori. Parte 2: sei classificatori con validazione leave-one-subject-out. Parte 3: grafici dei risultati. Parte 4: le due persone su cui il modello sbaglia.

## Fase 0 — Setup
Librerie, dataset ed elenco dei soggetti.

In [ ]:
import os
import pandas as pd
import numpy as np

if not os.path.exists("VitaStress"):
    !git clone https://github.com/paulvincenz/VitaStress.git

DATA = "VitaStress/data"
persone = sorted([c.replace("id_", "") for c in os.listdir(DATA) if c.startswith("id_")])
print("Persone:", len(persone))

## Fase 1 — Orari delle fasi
Dalle annotazioni ricava inizio e fine di ogni fase.

In [ ]:
fasi = {"baseline": "baseline", "cognitive": "cognitive",
        "public": "public speaking", "physical": "physical"}

def trova_orari(persona):
    cartella = DATA + "/id_" + persona + "/" + persona + "_"
    ann = pd.read_csv(cartella + "annotation.csv")
    ann["timestamp"] = pd.to_datetime(ann["timestamp"], format="mixed")
    ann["evento"] = ann["Button Name"].astype(str).str.lower().str.replace(":", "").str.strip(" []'")
    orari = {}
    for fase in fasi:
        nome = fasi[fase]
        inizio = ann[ann["evento"].str.startswith(nome + " start")]["timestamp"]
        fine = ann[ann["evento"].str.startswith(nome + " stop")]["timestamp"]
        if len(inizio) == 0 or len(fine) == 0:
            print(persona[:8], fase, "-> manca inizio o fine, salto")
            continue
        durata = (fine.iloc[0] - inizio.iloc[0]).total_seconds() / 60
        if durata <= 0 or durata > 30:
            print(persona[:8], fase, "-> durata strana:", round(durata, 1), "min, salto")
            continue
        orari[fase] = (inizio.iloc[0], fine.iloc[0])
    return orari

## Fase 2 — Segnali ogni 30 secondi
Media ogni 30 s per battito, temperatura e sudore; media, dev. standard e varianza per l'accelerometro.

In [ ]:
def leggi(cartella, file, colonna):
    dati = pd.read_csv(cartella + file + ".csv", usecols=["date", colonna])
    dati["date"] = pd.to_datetime(dati["date"], format="mixed")
    return dati.set_index("date")[colonna]

def segnali_30s(persona):
    cartella = DATA + "/id_" + persona + "/" + persona + "_"

    bpm = leggi(cartella, "activity", "bpm").replace(0, np.nan)
    temp = leggi(cartella, "temperature", "temp_sk1").replace(0, np.nan)
    sudore = leggi(cartella, "emography", "stress_skin")

    tab = pd.DataFrame({
        "bpm": bpm.resample("30s").mean(),
        "temp_sk1": temp.resample("30s").mean(),
        "stress_skin": sudore.resample("30s").mean().ffill(limit=1),
    })
    for asse in ["accX", "accY", "accZ"]:
        acc = leggi(cartella, "acc", asse).resample("30s")
        tab[asse + "_mean"] = acc.mean()
        tab[asse + "_std"] = acc.std()
        tab[asse + "_var"] = acc.var()
    return tab

## Fase 3 — Fasi ed etichetta
Tiene solo i blocchi dentro una fase. Riposo = 0, calcolo + discorso + corsa = 1.

In [ ]:
def tabella_persona(persona):
    tab = segnali_30s(persona)
    orari = trova_orari(persona)
    pezzi = []
    for fase in orari:
        inizio, fine = orari[fase]
        pezzo = tab[(tab.index >= inizio) & (tab.index < fine)].copy()
        pezzo["fase"] = fase
        pezzi.append(pezzo)
    tab = pd.concat(pezzi)
    tab["persona"] = persona
    tab["stress"] = (tab["fase"] != "baseline").astype(int)
    return tab

## Fase 4 — Tutti i soggetti
Circa 30 secondi di calcolo.

In [ ]:
tabella = pd.concat([tabella_persona(p) for p in persone]).reset_index()
print("Righe:", len(tabella))
print(tabella["fase"].value_counts())
tabella.head()

## Fase 5 — Dati mancanti
Percentuale di valori mancanti per colonna e fase.

In [ ]:
colonne = ["bpm", "stress_skin", "temp_sk1",
           "accX_mean", "accX_std", "accX_var", "accY_mean", "accY_std", "accY_var",
           "accZ_mean", "accZ_std", "accZ_var"]

mancanti = tabella.groupby("fase")[colonne].apply(lambda t: t.isna().mean() * 100).round(1)
mancanti.T

## Fase 6 — Pulizia e normalizzazione
Tolgo le righe incomplete, normalizzo ogni persona rispetto a sé stessa, salvo la tabella.

In [ ]:
prima = len(tabella)
tabella = tabella.dropna(subset=colonne)
print("Righe tolte:", prima - len(tabella), "-> rimaste", len(tabella))

for c in colonne:
    tabella[c] = tabella.groupby("persona")[c].transform(lambda s: (s - s.mean()) / s.std())

print(tabella["stress"].value_counts())
tabella.to_csv("tabella_30s.csv", index=False)

# Parte 2 — Classificatore binario

## Fase 7 — Feature ed etichetta
X = cosa il modello guarda, y = cosa deve indovinare, gruppi = la persona di ogni riga.

In [ ]:
X = tabella[colonne]
y = tabella["stress"]
gruppi = tabella["persona"]

print("Righe:", len(X), "| feature:", X.shape[1])
print("Rispondendo sempre 'stress':", round(y.mean() * 100, 1), "%")

## Fase 8 — I sei classificatori
Gli stessi del paper.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

modelli = {
    "LR": LogisticRegression(max_iter=1000),
    "RF": RandomForestClassifier(n_estimators=100, random_state=42),
    "DT": DecisionTreeClassifier(random_state=42),
    "LDA": LinearDiscriminantAnalysis(),
    "XGB": XGBClassifier(random_state=42),
    "kNN": KNeighborsClassifier(n_neighbors=5),
}

## Fase 9 — Validazione leave-one-subject-out
Addestra su 20 persone, prova sulla 21ª, ripete per tutte. Restituisce media e deviazione standard di accuracy e F1.

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score

def valuta(modello, X, y, gruppi):
    acc, f1 = [], []
    for train, test in LeaveOneGroupOut().split(X, y, gruppi):
        modello.fit(X.iloc[train], y.iloc[train])
        previsto = modello.predict(X.iloc[test])
        acc.append(accuracy_score(y.iloc[test], previsto))
        f1.append(f1_score(y.iloc[test], previsto, average="macro"))
    return np.mean(acc), np.std(acc), np.mean(f1), np.std(f1)

## Fase 10 — Risultati
Una riga per classificatore, da confrontare con la Tabella 4 del paper.

In [ ]:
risultati = []
for nome in modelli:
    a, a_sd, f, f_sd = valuta(modelli[nome], X, y, gruppi)
    risultati.append({"modello": nome,
                      "accuracy": f"{a:.2f} ± {a_sd:.2f}",
                      "F1": f"{f:.2f} ± {f_sd:.2f}",
                      "acc_media": a})

risultati = pd.DataFrame(risultati)
risultati

# Parte 3 — Grafici

## Fase 11 — Stile
Colori e stile comuni a tutti i grafici.

In [ ]:
import matplotlib.pyplot as plt

BLU, ARANCIO, GRIGIO = "#2a78d6", "#eb6834", "#898781"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#c3c2b7", "axes.grid": True, "axes.axisbelow": True,
                     "grid.color": "#e1e0d9", "grid.linewidth": 0.8})

## Fase 12 — Noi contro il paper
Accuracy media di ogni classificatore accanto a quella della Tabella 4.

In [ ]:
paper = {"LR": 0.88, "RF": 0.89, "DT": 0.88, "LDA": 0.88, "XGB": 0.87, "kNN": 0.87}
nomi = list(paper.keys())
noi = [risultati.loc[risultati["modello"] == n, "acc_media"].iloc[0] for n in nomi]
x = np.arange(len(nomi))

plt.figure(figsize=(9, 4.5))
plt.bar(x - 0.2, noi, width=0.38, color=BLU, edgecolor="white", linewidth=2, label="noi")
plt.bar(x + 0.2, [paper[n] for n in nomi], width=0.38, color=ARANCIO, edgecolor="white", linewidth=2, label="paper (Tabella 4)")
plt.axhline(0.65, color=GRIGIO, linestyle="--", linewidth=1.5, label="sempre «stress» (0,65)")
plt.xticks(x, nomi)
plt.ylim(0.5, 1)
plt.ylabel("accuracy")
plt.title("Accuracy per classificatore: noi contro il paper")
plt.legend(loc="lower left", framealpha=1)
plt.show()

## Fase 13 — Previsioni di Random Forest
Rifà la validazione solo per Random Forest, salvando la previsione di ogni riga e l'accuracy di ogni persona.

In [ ]:
previsioni = pd.Series(index=y.index, dtype=int)
acc_persona = {}
for train, test in LeaveOneGroupOut().split(X, y, gruppi):
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X.iloc[train], y.iloc[train])
    previsioni.iloc[test] = rf.predict(X.iloc[test])
    persona = gruppi.iloc[test[0]]
    acc_persona[persona[:8]] = accuracy_score(y.iloc[test], previsioni.iloc[test])

## Fase 14 — Accuracy per persona
Una barra per ogni persona esclusa: mostra su chi il modello funziona e su chi no.

In [ ]:
acc_persona = pd.Series(acc_persona).sort_values()
plt.figure(figsize=(10, 4.5))
plt.bar(acc_persona.index, acc_persona.values, color=BLU, edgecolor="white", linewidth=2)
plt.axhline(acc_persona.mean(), color="#0b0b0b", linewidth=1.5, label=f"media ({acc_persona.mean():.2f})")
plt.axhline(0.65, color=GRIGIO, linestyle="--", linewidth=1.5, label="sempre «stress» (0,65)")
plt.legend(loc="lower right", framealpha=1)
plt.xticks(rotation=60, ha="right")
plt.ylim(0.4, 1.02)
plt.ylabel("accuracy")
plt.title("Random Forest: accuracy sulla persona esclusa, una barra per persona")
plt.show()

## Fase 15 — Matrice di confusione
Quante righe indovinate e quanti errori, divisi per tipo.

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y, previsioni)
etichette = ["non stress", "stress"]
plt.figure(figsize=(5, 4.3))
plt.imshow(cm, cmap="Blues")
plt.grid(False)
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=14,
                 color="white" if cm[i, j] > cm.max() / 2 else "#0b0b0b")
plt.xticks([0, 1], etichette)
plt.yticks([0, 1], etichette)
plt.xlabel("previsto")
plt.ylabel("vero")
plt.title("Random Forest: matrice di confusione")
plt.show()

## Fase 16 — Importanza delle feature
Quanto Random Forest usa ogni feature per decidere.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X, y)
importanza = pd.Series(rf.feature_importances_, index=colonne).sort_values()
colori = [ARANCIO if c in ["bpm", "stress_skin", "temp_sk1"] else BLU for c in importanza.index]
plt.figure(figsize=(8, 5))
plt.barh(importanza.index, importanza.values, color=colori)
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color=ARANCIO, label="fisiologia"), Patch(color=BLU, label="movimento")], loc="lower right")
plt.xlabel("importanza")
plt.title("Random Forest: quanto conta ogni feature")
plt.show()

## Fase 17 — Importanza per gruppo
Somma dell'importanza per gruppo di feature. Le medie dell'accelerometro dicono come è orientato il polso (la postura).

In [ ]:
gruppi_feature = {
    "battito": ["bpm"],
    "sudore + temperatura": ["stress_skin", "temp_sk1"],
    "movimento: medie": ["accX_mean", "accY_mean", "accZ_mean"],
    "movimento: std e varianza": ["accX_std", "accX_var", "accY_std", "accY_var",
                                  "accZ_std", "accZ_var"],
}
for nome in gruppi_feature:
    print(nome, round(importanza[gruppi_feature[nome]].sum(), 2))

# Parte 4 — Le due persone difficili

## Fase 18 — Errori per fase
Per `ddca342b` e `a5e823ad`: righe indovinate (sum) sul totale della fase (count), usando le previsioni della Fase 13.

In [ ]:
tabella["previsto"] = previsioni
tabella["giusto"] = tabella["previsto"] == tabella["stress"]
tabella["p8"] = tabella["persona"].str[:8]

for p in ["ddca342b", "a5e823ad"]:
    righe = tabella[tabella["p8"] == p]
    print(p)
    print(righe.groupby("fase")["giusto"].agg(["sum", "count"]), "\n")

## Fase 19 — Orientamento del polso a riposo
Medie grezze (non normalizzate) dell'accelerometro durante il riposo. Da fermi l'accelerometro misura la gravità: il segno degli assi dice come è girato il braccialetto. Circa 30 secondi.

In [ ]:
grezzi = pd.concat([tabella_persona(p) for p in persone]).dropna(subset=colonne)
riposo = grezzi[grezzi["fase"] == "baseline"]
assi = ["accX_mean", "accY_mean", "accZ_mean"]
orientamento = riposo.groupby(riposo["persona"].str[:8])[assi].mean()
orientamento["gravità"] = np.sqrt((orientamento ** 2).sum(axis=1))
orientamento.round(0).sort_values("accX_mean")

## Fase 20 — Verifica su ddca342b
Addestro su tutte le altre persone e provo solo su `ddca342b`, con tre gruppi di feature. Se il problema è l'orientamento, togliendo le medie dell'accelerometro deve migliorare.

In [ ]:
fisiologia = ["bpm", "stress_skin", "temp_sk1"]
medie = ["accX_mean", "accY_mean", "accZ_mean"]
prove = {
    "tutte": colonne,
    "senza medie acc": [c for c in colonne if c not in medie],
    "solo fisiologia": fisiologia,
}

prova = tabella["p8"] == "ddca342b"
for nome in prove:
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(tabella.loc[~prova, prove[nome]], y[~prova])
    previsto = rf.predict(tabella.loc[prova, prove[nome]])
    riposi_giusti = ((previsto == 0) & (y[prova] == 0)).sum()
    print(nome, "| accuracy", round(accuracy_score(y[prova], previsto), 2),
          "| riposi giusti", riposi_giusti, "su", (y[prova] == 0).sum())

## Note
- Battito e temperatura a 0 → mancanti
- Sudore: da un valore ogni 60 s a uno ogni 30 s, riempiendo con il valore precedente
- Righe con dati mancanti tolte
- Classi sbilanciate: circa 65% stress, 35% riposo
- Iperparametri: valori standard, senza grid search
- `ddca342b`: braccialetto orientato al contrario (a riposo asse X positivo e asse Z negativo, per gli altri il contrario)
- `a5e823ad`: riposo di 82 minuti escluso; la normalizzazione usa solo le fasi di stress